# Term Mapping Sanity Check (SapBERT)
Re-evaluates every non-identity entry in the mapping dict using the live SapBERT embedder.  
Entries whose cosine similarity falls **below the 0.95 merge threshold** are flagged as potentially wrong.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from rapidfuzz.fuzz import ratio as fuzz_ratio

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

from wound_ontology.embedding.embedder import Embedder
from wound_ontology.utils.string_utils import normalize_term

MERGE_THRESHOLD = 0.95

def _string_sim(a, b):
    return fuzz_ratio(normalize_term(a), normalize_term(b)) / 100.0

/opt/anaconda3/envs/first_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cache_dir = ROOT / 'data' / 'cache'
path = sorted(cache_dir.glob('term_mapping_*.json'))[-1]
print(f'Mapping file : {path.name}')

with open(path) as f:
    mapping = json.load(f)

non_id = {term: canonical for term, canonical in mapping.items() if term != canonical}
print(f'Total entries        : {len(mapping)}')
print(f'Non-identity entries : {len(non_id)}')

Mapping file : term_mapping_2026-05-04_16-39.json
Total entries        : 22912
Non-identity entries : 935


In [3]:
unique = list(dict.fromkeys(t for pair in non_id.items() for t in pair))
print(f'Embedding {len(unique)} unique terms with SapBERT...')

embedder = Embedder()
embs = embedder.embed(unique)
idx = {t: i for i, t in enumerate(unique)}
print('Done.')

Embedding 1725 unique terms with SapBERT...
[Embedder] Loading SapBERT on cpu...


/opt/anaconda3/envs/first_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[Embedder] SapBERT ready.


Done.


In [4]:
rows = []
for term, canonical in non_id.items():
    cosine = float(np.dot(embs[idx[term]], embs[idx[canonical]]))
    rows.append({
        'term':      term,
        'canonical': canonical,
        'cosine':    round(cosine, 4),
        'str_sim':   round(_string_sim(term, canonical), 4),
    })

df = pd.DataFrame(rows).sort_values('cosine')

flagged = df[df['cosine'] < MERGE_THRESHOLD].reset_index(drop=True)
clean   = df[df['cosine'] >= MERGE_THRESHOLD]

print(f'FLAGGED  (cosine < {MERGE_THRESHOLD}) : {len(flagged)}')
print(f'CLEAN    (cosine >= {MERGE_THRESHOLD}) : {len(clean)}')

FLAGGED  (cosine < 0.95) : 559
CLEAN    (cosine >= 0.95) : 376


In [5]:
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 60)
flagged

,term,canonical,cosine,str_sim
0,Uncollaborative,Collaborative,0.3435,0.9286
1,Acellular,Cellular,0.3648,0.9412
2,Hematological Disorder,Dermatological Disorder,0.4020,0.9333
3,Radial,Radical,0.4107,0.9231
4,Emotion,Motion,0.4286,0.9231
...,...,...,...,...
554,Pressure Ulcer (PU) Risk Assessment,Pressure Ulcer Risk Assessment,0.9491,1.0000
555,Patient Quality of Life,Patient Quality of Life (QoL),0.9492,1.0000
556,Inflammatory Bowel Diseases (IBD),Inflammatory Bowel Disease,0.9496,0.9811
557,Gas Chromatography-Mass Spectrometry,Gas Chromatography-Mass Spectrometry (GC-MS),0.9497,1.0000


In [6]:
out = cache_dir / 'flagged_mappings.csv'
flagged.to_csv(out, index=False)
print(f'Saved {len(flagged)} flagged mappings to {out.name}')

Saved 559 flagged mappings to flagged_mappings.csv
